# Introduction To LangChain

In [1]:
import os
import warnings
from pathlib import Path

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


In [4]:
from langchain_openai import ChatOpenAI

model_str: str = "google/gemini-2.0-flash-001"
# Deterministic responses
llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str,
)

# Creative responses
creative_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),
    base_url=settings.OPENROUTER_URL,
    temperature=0.9,
    model=model_str,
)

# Image Generation
image_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model="openai/gpt-4o-mini",
)

<br>

- We will be taking an article draft and using LangChain to generate various useful items around this article. 

- We'll be creating:
    - An article title
    - An article description
    - Editor advice where we will insert an additional paragraph in the article
    - A thumbnail / hero image for our article.

Here we input our article to start with. Currently this is using an article from the Aurelio AI learning page.

In [5]:
article: str = """
The next chapter in one of football's most confusing careers is upon us - and with it comes the feeling that the potential he once 
had will never be fulfilled.

Joao Felix, who has joined Al-Nassr from Chelsea on a two-year contract in a deal worth up 
to £43.7m, remains the third most expensive transfer ever - in terms of initial fee - having 
cost Atletico Madrid £113m from Benfica at the age of 19 in 2019.

The Portugal forward has since gone on to play for European giants Barcelona, AC Milan and 
Chelsea - yet, since leaving home, has never scored more than 10 goals in a season.

Now, aged 25, he is off to Saudi Arabia. As Lisbon-based journalist Marcus Alves puts it: 
"The feeling back home is that Felix has officially given up on being a truly top-level "
"international player."

So what happened?

"It doesn't seem there is any turning point for him," said Spanish football journalist 
Guillem Balague.

"It drives coaches mad. They see the potential but he'll never fulfil it. It's a mental "
"thing. It's not that he's not bothered - but he's not listening."
'Pure art' - the start at Benfica
Having come through the Benfica academy, Felix became the youngest player in Benfica B history 
when he made his debut in Portugal's second tier aged 16.

He went on to make his first-team debut in August 2018 - and, put quite simply, he was 
brilliant, especially in the second half of the season.

Felix netted in the Lisbon derby against Sporting just a week after his introduction 
and became the youngest player to score a Europa League hat-trick in their quarter-final 
tie with Eintracht Frankfurt.

He ended the season with 20 goals in 43 games across all competitions, 15 of those coming 
in 26 league matches.

Benfica won the title and Felix was named young player of the year, named in the 
Portuguese league team of the season - and later that year won the Golden Boy award for the best player in Europe aged under 21.

"Those six months of him playing at Estadio da Luz regularly were by far the best "
"I've witnessed from a player in almost a decade in Portugal," said journalist Alves.

"It was pure art, a joy to watch. He seemed destined for the top.

"Back in mid-2019, when Cristiano Ronaldo arrived at the Portugal camp for the Nations 
League finals, I remember seeing a headline on TV that said 'Ronaldo joins Felix'. Felix, not the team.

"That was no joke - it was just how highly rated Felix was at that time."

Then Atletico Madrid came calling to complete one of the biggest transfers in history.
Awards at Atletico - but not enough to justify the fee
The four most expensive transfers to this day, sorted by British pounds and not euros, were all made between 2017 and 2019.

The top two were Paris St-Germain's signings of Neymar and Kylian Mbappe.

Number three is Atletico Madrid's £113m move for Felix to replace Antoine Griezmann, who they sold to Barcelona for £107.7m 
in the fourth biggest deal.

There he would link up with Atletico's Argentine boss Diego Simeone, who is legendary for how hard he makes his teams work.

"We should have seen what was coming," said Balague. "On one occasion earlier on in his time at Atletico, Simeone got really 
mad at him during a game, asked him to do certain things and you could see how Joao Felix was ignoring him.

"He basically ended up doing nothing like the stuff Simeone was asking him to do. From then on he started to come in and "
"out of the side and we started to hear stories about his lack of defensive commitment. But I think it goes deeper."

There were good times at Atletico, too, but not enough of them to justify that fee.

In three and a half seasons in (and out) of the Atletico team he scored 35 goals, as well as 16 assists, in 131 games.

In 2020-21 he was part of the Atleti squad who won their second La Liga title under Simeone.

But he only started 14 league games that season - and netted just three goals after Christmas in any competition.
The following campaign was worse for the team but better individually, as Atletico finished third but he was named 
the club's player of the season, with 10 goals in all competitions coming before a season-ending injury in April.

"After two and a half years of flattering to deceive, Joao Felix is finally starting to look capable of becoming 
one of the best players in the world," a BBC article at the time opened with.

But it was a false dawn.

"The cost to Atletico suggests there was a raw talent," said Balague.

"I think he's the last generation of players as kids who were told how brilliant they were, that do not appreciate the 
other side of it that you need – which is to work without the ball. Even with the ball he's not consistent enough."

Halfway through the next season, Felix wanted out.

"He is the biggest bet this club has taken in its history," Atletico chief executive officer Gil Marin said in December 
2022., external "I personally think he's a top talent, a world-class player.

"For reasons it isn't worth getting into - the relationship between him and the boss [Simeone], the minutes played, his 
motivation right now - it makes you think that the reasonable thing is that if there's an option that's good for 
the player, good for the club, we can look at it.

"I'd love him to stay personally, but I don't think that's the player's idea."
Chelsea and Barcelona loans don't change fortunes
Then came his first loan spell at Chelsea.

In January 2023 Graham Potter's Blues signed him on a six-month deal for a loan fee of £9.7m. At the same time 
Atleti extended his contract for a year to 2027.

Arsenal and Manchester United had also been linked to him.

Felix looked quite good on his Chelsea debut in their derby match against Fulham... until he was sent off for a 
lunging tackle on Kenny Tete in a 2-1 defeat.

He would only start another 13 games for the club once his three-match ban was over, plus six more off the bench, 
scoring four goals.

And that summer new Chelsea boss Mauricio Pochettino, who replaced Frank Lampard, who had in turn replaced Potter, 
decided he did not want Felix so no permanent deal was struck.

Felix subsequently returned to Atletico, but was reportedly seen arguing with sporting director Andrea Berta and made 
to train with the reserves.

Griezmann, the man he was signed to replace, was back at Atletico and even took his number seven shirt.
"""

### Preparing our Prompts

- LangChain comes with several prompt classes and methods for organizing or constructing our prompts.
- We will cover these in more detail in later examples, but for now we'll cover the essentials that we need here.
- Prompts for chat agents are at a minimum broken up into three components, those are:
    - **System prompt**: this provides the instructions to our LLM on how it must behave, what it's objective is, etc.
    - **User prompt**: this is a user written input.
    - **AI prompt**: this is the AI generated output. When representing a conversation, previous generations will be inserted back into the next prompt and become part of the broader chat history.

#### Example

```txt
You are a helpful AI assistant, you will do XYZ.    | SYSTEM PROMPT

User: Hi, what is the capital of Australia?         | USER PROMPT
AI: It is Canberra                                  | AI PROMPT
User: When is the best time to visit?               | USER PROMPT
```

- LangChain provides us with templates for each of these prompt types.
- By using templates we can insert different inputs to the template, modifying the prompt based on the provided inputs.


In [6]:
from langchain.prompts import HumanMessagePromptTemplate, SystemMessagePromptTemplate

system_prompt = SystemMessagePromptTemplate.from_template(
    "You are an AI assistant that helps generate article titles."
)

user_prompt = HumanMessagePromptTemplate.from_template(
    """You are tasked with creating a name for an article.
The article is here for you to examine:
<article>{article}</article>

The name should be based of the context of the article.
Be creative, but make sure the names are clear, catchy,
and relevant to the theme of the article.

Only output the article name, no other explanation or
text can be provided.""",
    input_variables=["article"],
)

console.print(user_prompt.format(article="Test String"))

HumanMessage(
    content='You are tasked with creating a name for an article.\nThe article is here for you to 
examine:\n<article>Test String</article>\n\nThe name should be based of the context of the article.\nBe creative, 
but make sure the names are clear, catchy,\nand relevant to the theme of the article.\n\nOnly output the article 
name, no other explanation or\ntext can be provided.',
    additional_kwargs={},
    response_metadata={}
)

In [7]:
from langchain.prompts import ChatPromptTemplate

first_prompt = ChatPromptTemplate.from_messages(messages=[system_prompt, user_prompt])
console.print(first_prompt.format(article="Test String"))

System: You are an AI assistant that helps generate article titles.
Human: You are tasked with creating a name for an article.
The article is here for you to examine:
<article>Test String</article>

The name should be based of the context of the article.
Be creative, but make sure the names are clear, catchy,
and relevant to the theme of the article.

Only output the article name, no other explanation or
text can be provided.

<br>

- We can chain together our first_prompt template and the llm object we defined earlier to create a simple LLM chain.
- This chain will perform the steps prompt formatting > llm generation > get output.

- We'll be using `LangChain Expression Language (LCEL)` to construct our chain.
- This syntax can look a little strange but we will cover it in detail later in the course. - For now, all we need to know is that we define our inputs with the first dictionary segment `(ie {"article": lambda x: x["article"]})` and then we use the pipe operator `(|)` to say that the output from the left of the pipe will be fed into the input to the right of the pipe.

In [8]:
chain_one = (
    {"article": lambda x: x["article"]}
    | first_prompt  # pass {"article": article} to the prompt
    | creative_llm  # generate the response using the first_prompt
    | {"article_title": lambda x: x.content}  # extract the content from the response
)

In [9]:
article_title_msg = chain_one.invoke({"article": article})
console.print(article_title_msg)

{'article_title': 'Joao Felix: From Golden Boy to Saudi Arabia – a career unfulfilled?\n'}

- But we will actually chain this step with multiple other LLMChain steps.
- So, to continue, our next step is to summarize the article using both the article and newly generated article_title values, from which we will output a new summary variable

In [10]:
template: str = """You are tasked with creating a description for
the article. The article to examine:

<article>{article}</article>

<article_title>{article_title}</article_title>.

Output the SEO friendly article description. Do not output
anything other than the description."""

second_user_prompt = HumanMessagePromptTemplate.from_template(
    template=template,
    input_variables=["article", "article_title"],
)

second_prompt = ChatPromptTemplate.from_messages([system_prompt, second_user_prompt])

In [11]:
chain_two = (
    {"article": lambda x: x["article"], "article_title": lambda x: x["article_title"]}
    | second_prompt
    | llm
    | {"article_description": lambda x: x.content}
)

In [12]:
article_description_msg = chain_two.invoke(
    {"article": article, "article_title": article_title_msg["article_title"]}
)
console.print(article_description_msg)

{
    'article_description': "A look at Joao Felix's career, from his promising start at Benfica to his move to 
Al-Nassr, and why he has not lived up to his potential.\n"
}

In [13]:
template: str = """You are tasked with creating a new paragraph for the
article. The article to examine:

<article>{article}</article>

Choose one paragraph to review and edit. During your edit
ensure you provide constructive feedback to the user so they
can learn where to improve their own writing."""

third_user_prompt = HumanMessagePromptTemplate.from_template(
    template=template,
    input_variables=["article"],
)

# prompt template 3: creating a new paragraph for the article
third_prompt = ChatPromptTemplate.from_messages([system_prompt, third_user_prompt])

In [14]:
from pydantic import BaseModel, Field


class Paragraph(BaseModel):
    original_paragraph: str = Field(description="The original paragraph")
    edited_paragraph: str = Field(description="The improved edited paragraph")
    feedback: str = Field(
        description=("Constructive feedback on the original paragraph")
    )


structured_llm = creative_llm.with_structured_output(Paragraph)

In [15]:
# chain 3: inputs: article / output: article_para
chain_three = (
    {"article": lambda x: x["article"]}
    | third_prompt
    | structured_llm
    | {
        "original_paragraph": lambda x: x.original_paragraph,
        "edited_paragraph": lambda x: x.edited_paragraph,
        "feedback": lambda x: x.feedback,
    }
)

In [16]:
out = chain_three.invoke({"article": article})
console.print(out)

{
    'original_paragraph': "A subsequent loan to Barcelona was next. But like Chelsea, it didn't stick. He struggled
to find consistency and a permanent place in the starting eleven. The loan concluded and he was once again at a 
career crossroads.",
    'edited_paragraph': 'A subsequent loan to Barcelona offered another chance for Felix to reignite his career. 
However, much like his time at Chelsea, the move failed to deliver the expected impact. Despite flashes of 
brilliance, he struggled to find consistency and secure a permanent place in the starting eleven. The loan 
concluded with a sense of unfulfilled potential, leaving Felix once again at a career crossroads.',
    'feedback': "The original paragraph is a bit fragmented and lacks a strong concluding statement. It lists 
events without really connecting them to a central theme. In the edited paragraph, I've aimed to create a more 
cohesive narrative by emphasizing the recurring theme of unfulfilled potential and lack of consistency. I've also 
added a more definitive ending to provide a sense of closure to that particular part of his career."
}

In [17]:
from langchain_community.utilities.dalle_image_generator import DallEAPIWrapper
from langchain_core.prompts import PromptTemplate

image_prompt = PromptTemplate(
    input_variables=["article"],
    template=(
        "Generate a prompt with less than 500 characters to generate an image based on the following article: {article}"
    ),
)

In [18]:
import matplotlib.pyplot as plt
from langchain_core.runnables import RunnableLambda
from skimage import io


def generate_and_display_image(image_prompt: str) -> None:
    """
    Generate and display an image based on a prompt using DALL-E.

    Parameters
    ----------
    image_prompt : str
        The prompt describing the image to generate.

    Returns
    -------
    None
        Displays the generated image using matplotlib.

    Notes
    -----
    This function uses DallEAPIWrapper to generate an image URL from the prompt,
    loads the image using skimage.io, and displays it with matplotlib.
    """
    image_url = DallEAPIWrapper().run(image_prompt)
    image_data = io.imread(image_url)

    plt.imshow(image_data)
    plt.axis("off")
    plt.show()


# Wrap this in a RunnableLambda for use with LCEL
image_gen_runnable = RunnableLambda(generate_and_display_image)

In [20]:
# chain 4: inputs: article, article_para / outputs: new_suggestion_article
chain_four = (
    {"article": lambda x: x["article"]}
    | image_prompt
    | image_llm  # generate the image prompt using the image_prompt
    | (lambda x: x.content)
    | image_gen_runnable
)

In [ ]:
# This code block will fail if you don't have the OpenAI API keys
# or if the image generation fails.
try:
    chain_four.invoke({"article": article})
except Exception as e:
    print(f"Error occurred: {e}")

Error occurred: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable


AIMessage(content='Why did Tyrion Lannister bring a ladder to the bar?\n\nBecause he heard the drinks were on the house!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 17, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'gen-1753811387-0tpgnWs6O0VWMDPLCxn9', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--22659220-6142-42e6-840a-48856130a3bb-0', usage_metadata={'input_tokens': 17, 'output_tokens': 24, 'total_tokens': 41, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}})